# Visualización de métricas por baseline

Este notebook carga `storage/logs/metrics.jsonl`, lee los rangos definidos en `data/metrics_baselines.json`,
asigna cada fila a un baseline según la fecha, y genera una comparación visual entre:
- `r7-fidelity-v2`
- `septiembre-1`
- `septiembre-13`

Puedes ejecutar cada celda en orden para explorar las métricas y los gráficos.

In [2]:
from pathlib import Path
import json
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

METRICS_FILE = Path('storage/logs/metrics.jsonl')
BASELINES_FILE = Path('data/metrics_baselines.json')

def parse_ts(value):
    if not value:
        return None
    try:
        return datetime.fromisoformat(value)
    except Exception:
        return None

def load_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows

def load_baselines(path):
    if not path.exists():
        return []
    try:
        data = json.loads(path.read_text(encoding='utf-8-sig'))
    except Exception:
        return []
    return data.get('baselines', [])

def assign_baseline(row, baselines):
    row_ts = parse_ts(row.get('timestamp') or row.get('finished_at') or row.get('end_time'))
    if row_ts is None:
        return None

    for baseline in baselines:
        start_ts = parse_ts(baseline.get('started_at'))
        end_ts = parse_ts(baseline.get('ended_at'))

        if start_ts is None:
            continue
 
        if row_ts < start_ts:
            continue

        if end_ts is not None and row_ts > end_ts:
            continue

        return baseline.get('id')

    return None

metrics_rows = load_jsonl(METRICS_FILE)
baselines = load_baselines(BASELINES_FILE)

print(f'Métricas cargadas: {len(metrics_rows)} filas')
print(f'Baselines cargadas: {[b.get(id) for b in baselines]}')

df = pd.DataFrame(metrics_rows)
for col in ['llm_ms', 'retrieval_ms', 'fidelity_ms', 'total_ms', 'tokens_est']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['assigned_baseline'] = df.apply(lambda row: assign_baseline(row, baselines), axis=1)
df.head()

Métricas cargadas: 862 filas
Baselines cargadas: [None, None, None]


,timestamp,route,intent_type,channel,retrieval_ms,llm_ms,tokens_est,cached,num_docs,total_ms,baseline_id,metric_schema_version,fidelity_ms,fidelity_status,assigned_baseline
0,2026-06-12T22:01:17,memory,work_state,cli,0,25589,27,False,0,0.0,NaN,NaN,0.0,NaN,None
1,2026-06-12T22:02:31,memory,tasks,cli,0,2,42,False,0,0.0,NaN,NaN,0.0,NaN,None
2,2026-06-12T22:03:06,tool_save_fact,tool_save_fact,cli,0,4,0,False,0,0.0,NaN,NaN,0.0,NaN,None
3,2026-06-12T22:03:18,tool_create_task,tool_create_task,cli,0,5,0,False,0,0.0,NaN,NaN,0.0,NaN,None
4,2026-06-12T22:03:28,tool_create_task,tool_create_task,cli,0,11,0,False,0,0.0,NaN,NaN,0.0,NaN,None


In [1]:
summary_rows = []

for baseline in baselines:
    baseline_id = baseline.get('id')
    rows = df[df['assigned_baseline'] == baseline_id]

    summary_rows.append({
        'baseline_id': baseline_id,
        'turnos': len(rows),
        'avg_total_s': rows['total_ms'].mean() / 1000 if len(rows) else 0,
        'avg_llm_s': rows['llm_ms'].mean() / 1000 if len(rows) else 0,
        'avg_retrieval_s': rows['retrieval_ms'].mean() / 1000 if len(rows) else 0,
        'avg_fidelity_s': rows['fidelity_ms'].mean() / 1000 if len(rows) else 0,
        'tokens_est': int(rows['tokens_est'].sum()) if len(rows) else 0,
    })

summary = pd.DataFrame(summary_rows)
summary

NameError: name 'baselines' is not defined

In [3]:
melt = summary.melt(
    id_vars='baseline_id',
    value_vars=['avg_total_s', 'avg_llm_s', 'avg_retrieval_s', 'avg_fidelity_s'],
    var_name='metric',
    value_name='seconds'
)

plt.figure(figsize=(12, 6))
sns.barplot(data=melt, x='baseline_id', y='seconds', hue='metric')
plt.title('Comparación de tiempos promedio por baseline')
plt.ylabel('Segundos')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

NameError: name 'summary' is not defined

In [4]:
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x='baseline_id', y='tokens_est')
plt.title('Tokens consumidos por baseline')
plt.ylabel('Tokens totales')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

NameError: name 'summary' is not defined

<Figure size 800x500 with 0 Axes>